In [7]:
import cisei_lib.core.rf.rssi_core as rc
import math
from pathlib import Path

# 1. REMOVE INVALID RECORDS

current_dir = Path.cwd()
base_path = Path(current_dir.parent / 'data')

def to_path(file):
    return Path(current_dir.parent / 'data' / file )

features_file = to_path('all_chunks.json')
print(features_file)

records = rc.load_json(features_file)

print('input records', len(records))

records = [
    r for r in records
    if isinstance(r.get("features"), dict)
]

rules = [
    {"feature_path": "measures.rssi", "min_value": -120, "max_value": -35, "include_max": True},
    {"feature_path": "tx.pw", "min_value": 0, "max_value": 40, "include_max": True},
    {"feature_path": "tx.ant_gain", "min_value": 0, "max_value": 40, "include_max": True},
    {"feature_path": "rx.ant_gain", "min_value": 0, "max_value": 40, "include_max": True},
    # {"feature_path": "features.tx_near_terminal_clearance_m", "min_value": -20, "max_value": 20, "include_max": True}
]

clean = rc.filter_records_by_many(records, rules)
print("clean:", len(clean))


/workspaces/planning_service/tests/data/all_chunks.json
input records 2468
clean: 1890


In [9]:
# 2. ADJUST NO CLUTTER MODEL

rules = [
    {"feature_path": "features.vegetation.boundary", "min_value": 0, "max_value": 1, "include_max": False},
    {"feature_path": "features.buildings.boundary", "min_value": 0, "max_value": 1, "include_max": False},
    {"feature_path": "features.terrain.boundary", "min_value": 0, "max_value": 1, "include_max": False}
]

no_clutter = rc.filter_records_by_many(clean, rules)
print("build_tail:", len(no_clutter))


base_model = rc.fit_expression_model(
    no_clutter,
    fixed_terms={
        "tx_power": "tx.pw",
        "tx_gain": "tx.ant_gain",
        "rx_gain": "rx.ant_gain"
    },
    fixed_coeffs={
        "tx_power": 1.0,
        "tx_gain": 1.0,
        "rx_gain": 1.0,
    },
    learned_terms={
        "fspl": "-features.fspl",
        "diff": "-features.delta_diffra"
    },
    learned_priors={
        "fspl": 1.0,
        "diff": 1.0,
    },
    bounds={
        "fspl": (0.0, None),
        "diff": (0.0, None)       
    },
    y_expr="measures.rssi",
    fit_intercept=False,
    tau=0.5, # increase to make the model optimistic
    l2_to_prior=0.01,
)

print("coef corrected:", base_model.learned_coeffs)

base_model.save(to_path('base_model.json'))

#---------------------------------------------------------------------------------------------------------
# VEGETATION EVALUATION
#---------------------------------------------------------------------------------------------------------
rules = [
    {"feature_path": "features.vegetation.boundary", "min_value": 10, "max_value": 100000, "include_max": True},
    {"feature_path": "features.buildings.boundary", "min_value": 0, "max_value": 10, "include_max": False},
    {"feature_path": "features.terrain.fresnel", "min_value": 0, "max_value": 10, "include_max": False}
]

veg_clutter = rc.filter_records_by_many(clean, rules)
print("veg_clutter:", len(veg_clutter))

base_weights = base_model.learned_coeffs
veg_model = rc.fit_expression_model(
    veg_clutter,
    fixed_terms={
        "tx_power": "tx.pw",
        "tx_gain": "tx.ant_gain",
        "rx_gain": "rx.ant_gain",
        "fspl": "-features.fspl",
        "diff": "-features.delta_diffra",
    },
    fixed_coeffs={
        "tx_power": 1.0,
        "tx_gain": 1.0,
        "rx_gain": 1.0,
        "fspl": base_weights['fspl'],
        "diff": base_weights['diff'],
    },
    learned_terms={    
        "veg_boundary": "-features.vegetation.boundary**0.25",
        "veg_fresnel":  "-features.vegetation.fresnel**0.25",
        "veg_core":     "-features.vegetation.core**0.25",

    },
    learned_priors={   
        "veg_boundary": 1,     
        "veg_fresnel": 1,
        "veg_core": 1,
    },
    bounds={
        "veg_boundary": (0.0, None),
        "veg_fresnel": (0.0, None),
        "veg_core": (0.0, None),
    },
    y_expr="measures.rssi", # _model.env_excess_loss
    fit_intercept=False,
    tau=0.25,
    l2_to_prior=0.0001,
)

print("coef corrected:", veg_model.learned_coeffs)
veg_model.save(to_path('veg_model.json'))

#---------------------------------------------------------------------------------------------------------
# BUILDING EVALUATION
#---------------------------------------------------------------------------------------------------------

rules = [
    {"feature_path": "features.vegetation.boundary", "min_value": 0, "max_value": 1, "include_max": True},
    {"feature_path": "features.buildings.boundary", "min_value": 10, "max_value": 1000, "include_max": False},
    {"feature_path": "features.terrain.fresnel", "min_value": 0, "max_value": 1, "include_max": True}
]

bldg_clutter = rc.filter_records_by_many(clean, rules)
print("bldg_clutter:", len(bldg_clutter))

base_weights = base_model.learned_coeffs
bldg_model = rc.fit_expression_model(
    bldg_clutter,
    fixed_terms={
        "tx_power": "tx.pw",
        "tx_gain": "tx.ant_gain",
        "rx_gain": "rx.ant_gain",
        "fspl": "-features.fspl",
        "diff": "-features.delta_diffra",
    },
    fixed_coeffs={
        "tx_power": 1.0,
        "tx_gain": 1.0,
        "rx_gain": 1.0,
        "fspl": base_weights['fspl'],
        "diff": base_weights['diff'],
    },
    learned_terms={    
    "bldg_boundary": "-math.log10(1.0 + features.buildings.boundary)",
    "bldg_fresnel":  "-math.log10(1.0 + features.buildings.fresnel)",
    "bldg_core":     "-math.log10(1.0 + features.buildings.core)",
    },
    learned_priors={   
        "bldg_boundary": 1,     
        "bldg_fresnel": 1,
        "bldg_core": 1,
    },
    bounds={
        "bldg_boundary": (0.0, None),
        "bldg_fresnel": (0.0, None),
        "bldg_core": (0.0, None),
    },
    y_expr="measures.rssi", # _model.env_excess_loss
    fit_intercept=False,
    tau=0.25,
    l2_to_prior=0.0001,
)

print("coef corrected:", bldg_model.learned_coeffs)
bldg_model.save(to_path('bldg_model.json'))



build_tail: 243
coef corrected: {'fspl': 1.114337839919779, 'diff': 1.0}
veg_clutter: 188
coef corrected: {'veg_boundary': 3.636938492774212, 'veg_fresnel': 3.361691875766035, 'veg_core': 2.1605049864827635}
bldg_clutter: 521
coef corrected: {'bldg_boundary': 3.0128246563174357, 'bldg_fresnel': 2.0746410649095632, 'bldg_core': 1.3730842851038247}


In [10]:
# FULL MODEL TRAINING

base_model = rc.ExpressionModel.load(to_path('base_model.json'))
base_weights = base_model.learned_coeffs
veg_model = rc.ExpressionModel.load(to_path('veg_model.json'))
veg_weights = veg_model.learned_coeffs
print(veg_weights)
bldg_model = rc.ExpressionModel.load(to_path('bldg_model.json'))
bldg_weights = bldg_model.learned_coeffs
print(bldg_weights)

training = [
    r for r in clean
    if rc.eval_expr(r, "measures.rssi") is not None
    and rc.eval_expr(r, "measures.rssi") < -70    
    and rc.eval_expr(r, "measures.rssi") > -120
]

final_model = rc.fit_expression_model(
    clean,
    fixed_terms={
        "tx_power": "tx.pw",
        "tx_gain": "tx.ant_gain",
        "rx_gain": "rx.ant_gain",
        "fspl": "-features.fspl",
        "diff": "-features.delta_diffra",
    },
    fixed_coeffs={
        "tx_power": 1.0,
        "tx_gain": 1.0,
        "rx_gain": 1.0,
        "fspl": base_weights["fspl"],
        "diff": base_weights["diff"],
    },
    learned_terms={

        "veg_boundary": "-features.vegetation.boundary**0.25",
        "veg_fresnel":  "-features.vegetation.fresnel**0.25",
        "veg_core":     "-features.vegetation.core**0.25",
        
        
        "bldg_boundary": "-math.log10(1.0 + features.buildings.boundary)",
        "bldg_fresnel":  "-math.log10(1.0 + features.buildings.fresnel)",
        "bldg_core":     "-math.log10(1.0 + features.buildings.core)",

    },
    learned_priors={
        "veg_boundary": veg_weights['veg_boundary'],
        "veg_fresnel": veg_weights['veg_fresnel'],
        "veg_core": veg_weights['veg_core'],

        "bldg_boundary": bldg_weights['bldg_boundary'],
        "bldg_fresnel": bldg_weights['bldg_fresnel'],
        "bldg_core": bldg_weights['bldg_core'],

    },
    bounds={
        "veg_boundary": (0, None),
        "veg_fresnel": (0, None),
        "veg_core": (0, None),

        "bldg_boundary": (0.0, None),
        "bldg_fresnel": (0.0, None),
        "bldg_core": (0.0, None),

    },
    fit_intercept=False,
    tau=0.5,
    l2_to_prior=0.0001,
)

print(final_model.learned_coeffs)

final_model.save(to_path('final_model.json'))

{'veg_boundary': 3.636938492774212, 'veg_fresnel': 3.361691875766035, 'veg_core': 2.1605049864827635}
{'bldg_boundary': 3.0128246563174357, 'bldg_fresnel': 2.0746410649095632, 'bldg_core': 1.3730842851038247}
{'veg_boundary': 2.3734131419421973, 'veg_fresnel': 2.3203079642728555, 'veg_core': 1.4750061267745072, 'bldg_boundary': 1.9062843348847418, 'bldg_fresnel': 1.1755094720633916, 'bldg_core': 1.0576422786213882}


In [4]:
base_model = rc.ExpressionModel.load(to_path('base_model.json'))
final_model = rc.ExpressionModel.load(to_path('final_model.json'))

border_line = [
    r for r in clean
    if rc.eval_expr(r, "measures.rssi") is not None
    and rc.eval_expr(r, "measures.rssi") < -70
    and rc.eval_expr(r, "measures.rssi") > -80
]

bad = [
    r for r in clean
    if rc.eval_expr(r, "measures.rssi") is not None
    and rc.eval_expr(r, "measures.rssi") < -80
    and rc.eval_expr(r, "measures.rssi") > -120 ]


datasets = zip(('all', 'borderline', 'bad'), (clean, border_line, bad))

for dataset in datasets:

    print('\nEvaluation of:', dataset[0], 'with:', len(dataset[1]), 'elements')
    print('final_model:', rc.evaluate_model(final_model, dataset[1]))
    print('base_model:', rc.evaluate_model(base_model, dataset[1]))

    res1 = rc.describe_residuals(final_model, dataset[1])
    res2 = rc.describe_residuals(base_model, dataset[1])


    print('\nTop (25%) optimistic and pessimistic errors')
    print('final model:', 'Optimism:', res1['25%'], 'Pessimism', res1['75%'])
    print('base_model:', 'Optimism:', res2['25%'], 'Pessimism', res2['75%'])




Evaluation of: all with: 1890 elements
final_model: {'n': 1890, 'mae': 8.635865305532995, 'rmse': 11.51522443112991, 'bias': 1.5910807733932666, 'r2': -0.014337333609322211}
base_model: {'n': 1890, 'mae': 9.251155287828654, 'rmse': 12.187449643376876, 'bias': -5.133408066319618, 'r2': -0.1362221867996054}

Top (25%) optimistic and pessimistic errors
final model: Optimism: -5.443984921720434 Pessimism 7.965301495990076
base_model: Optimism: -12.583426725897807 Pessimism 2.501572525123189

Evaluation of: borderline with: 518 elements
final_model: {'n': 518, 'mae': 8.507658910548628, 'rmse': 11.259301329160705, 'bias': -1.6827866947552736, 'r2': -15.217508197223665}
base_model: {'n': 518, 'mae': 10.961455410667796, 'rmse': 13.046850054553044, 'bias': -9.395308282320993, 'r2': -20.775723031172003}

Top (25%) optimistic and pessimistic errors
final model: Optimism: -9.019476692094615 Pessimism 3.800947072558621
base_model: Optimism: -15.747584373035329 Pessimism -3.988042479018951

Evaluat

In [11]:
import json

rows = []

for i, r in enumerate(clean):   # or clean, borderline, bad
    exp = final_model.explain(r)
    if exp is None or exp["residual"] is None:
        continue

    rows.append({
        "idx": i,
        "measured": exp["measured"],
        "predicted": exp["prediction"],
        "residual": exp["residual"],
        "abs_error": abs(exp["residual"]),
        "record": r,
        "explain": exp,
    })

worst = sorted(rows, key=lambda x: x["abs_error"], reverse=True)[:50]

with open(to_path("worst_eval.json"), "w", encoding="utf-8") as f:
    json.dump(worst, f, indent=4)  # indent=4 makes it readable
    

In [12]:
print(worst[0]['residual'])
print(worst[49]['residual'])


64.48661558829693
27.141513109507073


In [13]:
worst[0]


{'idx': 44,
 'measured': -56.0,
 'predicted': -120.48661558829693,
 'residual': 64.48661558829693,
 'abs_error': 64.48661558829693,
 'record': {'rx': {'name': 'BIT-R-GE-023',
   'pos': [-26.152901, -51.54521],
   'ant_type': 'YAGI',
   'ant_gain': 14.0,
   'ant_height': 7,
   'mode': 'remote',
   'state': 'high-sensitivity',
   'rate': 'auto',
   'hops': 1.0},
  'tx': {'name': 'BIT-R-P7-001',
   'pos': [-26.13643737, -51.55550663],
   'ant_type': 'OMNI',
   'ant_gain': 8.15,
   'ant_height': 0.0,
   'mode': 'access-point',
   'state': 'high-sensitivity',
   'rate': '500kbps',
   'pw': 30.0,
   'retry': 0.0,
   'hops': 0},
  'measures': {'lqi': 8.5, 'rssi': -56.0, 'snr': 52.9, 'loss_rate': 3.0},
  'features': {'fspl': 99.77,
   'dist_m': 2094.54,
   'delta_diffra': 37.69,
   'terrain': {'core': 240, 'fresnel': 277.5, 'boundary': 274.38},
   'vegetation': {'core': 130, 'fresnel': 66.25, 'boundary': 88.12},
   'buildings': {'core': 0, 'fresnel': 27.98, 'boundary': 45.48},
   'terrain_peak